<a href="https://colab.research.google.com/github/Jaat2727/Torrent-Downloader-on-Collab-To-Google-Drive/blob/main/Best_Simple_TORRENT_DOWNLOADER_updated%20with%20ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 📂 Install Libraries & Mount Google Drive
#@markdown Run this cell to:
#@markdown 1. Install **aria2** and its Python API wrapper for clean progress bars.
#@markdown 2. Connect and mount your Google Drive to `/content/drive`.

import os
import subprocess
import time
import sys

print("--- Starting Setup ---", flush=True)

# 1. Install aria2 and aria2p
print("\n⚙️ Installing aria2 & aria2p...", flush=True)
try:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'aria2p', '-q'], check=True)
    print("✅ aria2 & wrapper installed successfully.")
except Exception as e:
    print(f"❌ ERROR during installation: {e}", flush=True)

# 2. Mount Google Drive
print("\n⚙️ Mounting Google Drive...", flush=True)
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
        time.sleep(2)
        if os.path.exists('/content/drive/MyDrive'):
             print("✅ Google Drive mounted successfully.")
        else:
             print("⚠️ Mount attempt finished, but '/content/drive/MyDrive' not found.", flush=True)
    else:
        print("✅ Google Drive already mounted.")
except Exception as e:
    print(f"❌ An error occurred during mounting: {e}", flush=True)

print("\n--- Setup Finished ---", flush=True)

--- Starting Setup ---

⚙️ Installing aria2 & aria2p...
✅ aria2 & wrapper installed successfully.

⚙️ Mounting Google Drive...
✅ Google Drive already mounted.

--- Setup Finished ---


.

.

.

.

In [2]:
#@title 🧲 Google Drive Torrent Downloader (Ultra-Fast Transfer Mode)
import os
import subprocess
import time
import shutil
from IPython.display import display, clear_output, HTML

#@markdown **Enter Magnet Link:**
MAGNET_LINK = "your  magnet_link  place it here" #@param {type:"string"}

#@markdown **Settings:**
DOWNLOAD_PATH = "/content/drive/MyDrive/MyDownloads" #@param {type:"string"}
#@markdown *Use Colab's SSD as a cache for 10x faster downloads. (Uncheck this if your torrent is massive and causes 'Disk Full' errors).*
USE_FAST_SSD_CACHE = True #@param {type:"boolean"}

def format_bytes(size):
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size < 1024.0:
            return f"{size:.2f} {unit}"
        size /= 1024.0
    return f"{size:.2f} PB"

if not MAGNET_LINK or MAGNET_LINK == "your  magnet_link  place it here":
    print("⚠️ Please enter a Magnet Link in the form above.", flush=True)
elif '/drive/' in DOWNLOAD_PATH and not os.path.exists('/content/drive/MyDrive'):
    print("❌ Google Drive does not appear to be mounted. Please run the mounting cell first.", flush=True)
else:
    # 1. SETUP PATHS
    os.makedirs(DOWNLOAD_PATH, exist_ok=True)

    if USE_FAST_SSD_CACHE and '/drive/' in DOWNLOAD_PATH:
        # Create a unique temporary local path on the lightning-fast SSD
        LOCAL_TMP_PATH = f"/content/Colab_Temp_Download_{int(time.time())}"
        os.makedirs(LOCAL_TMP_PATH, exist_ok=True)
    else:
        LOCAL_TMP_PATH = DOWNLOAD_PATH

    try:
        import aria2p
    except ImportError:
        print("❌ aria2p is not installed. Please run the setup cell first.")
        import sys
        sys.exit(1)

    # 2. START ARIA2c DAEMON
    subprocess.run(["pkill", "-f", "aria2c"])
    time.sleep(1)

    cmd = [
        "aria2c",
        "--enable-rpc",
        "--rpc-listen-all=false",
        "--rpc-listen-port=6800",
        "--seed-time=0",
        "--file-allocation=none",
        "--max-connection-per-server=16",
        "--min-split-size=1M",
        "--bt-max-peers=200",
        "--bt-request-peer-speed-limit=10M",
        "--split=16",
        "-D"
    ]
    subprocess.Popen(cmd)
    time.sleep(2)

    aria2 = aria2p.API(aria2p.Client(host="http://localhost", port=6800, secret=""))

    try:
        download = aria2.add_magnet(MAGNET_LINK, options={'dir': LOCAL_TMP_PATH})
    except Exception as e:
         print(f"❌ Failed to add magnet link: {e}")
         import sys
         sys.exit(1)

    print("⏳ Connecting to peers... (This might take a moment)", flush=True)
    while not download.name and not download.has_failed:
        time.sleep(1)
        download.update()

    if download.has_failed:
        print(f"❌ Download failed! Error: {download.error_message}")
    else:
        # 3. DOWNLOAD UI LOOP
        while True:
            download.update()

            # Catch Metadata to Torrent transition
            if download.is_complete:
                all_downloads = aria2.get_downloads()
                active_downloads = [d for d in all_downloads if not d.is_complete]

                if download.followed_by_ids:
                    download = aria2.get_download(download.followed_by_ids[0])
                    continue
                elif active_downloads:
                    download = active_downloads[0]
                    continue
                else:
                    break

            if download.has_failed:
                break

            progress = download.progress
            speed_str = download.download_speed_string()
            peers = download.connections
            eta_str = download.eta_string()
            status = download.status
            name = download.name

            total_size_str = format_bytes(download.total_length)
            downloaded_str = format_bytes(download.completed_length)

            file_rows_html = ""
            for f in download.files:
                fname = os.path.basename(f.path)
                if not fname or fname.endswith('.torrent'):
                    fname = "Torrent Metadata"

                f_size_str = format_bytes(f.length)
                f_prog = (f.completed_length / f.length * 100) if f.length > 0 else 0
                bar_bg = "linear-gradient(90deg, #4CAF50, #8BC34A)" if f_prog == 100 else "linear-gradient(90deg, #005A9E, #00A4EF)"

                file_rows_html += f'''
                <tr style="background-color: #1c1c1c;">
                    <td style="padding: 10px; border-radius: 6px 0 0 6px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; max-width: 300px; border-bottom: 1px solid #2a2a2a;">{fname}</td>
                    <td style="padding: 10px; text-align: right; border-bottom: 1px solid #2a2a2a; color: #999;">{f_size_str}</td>
                    <td style="padding: 10px; border-radius: 0 6px 6px 0; border-bottom: 1px solid #2a2a2a;">
                        <div style="display: flex; align-items: center; justify-content: flex-end;">
                            <span style="font-size: 11px; margin-right: 10px; color: #888; width: 40px; text-align: right;">{f_prog:.1f}%</span>
                            <div style="background-color: #0a0a0a; width: 80px; height: 6px; border-radius: 3px; overflow: hidden; border: 1px solid #333;">
                                <div style="background: {bar_bg}; width: {f_prog}%; height: 100%;"></div>
                            </div>
                        </div>
                    </td>
                </tr>
                '''

            ui_html = f"""
            <div style="background-color: #121212; color: #f5f5f5; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; width: 100%; max-width: 800px; border: 1px solid #333; border-radius: 12px; overflow: hidden; box-shadow: 0 8px 16px rgba(0,0,0,0.5);">
                <div style="background-color: #1a1a1a; padding: 20px; border-bottom: 1px solid #222;">
                    <h3 style="margin: 0 0 15px 0; color: #fff; font-size: 18px; font-weight: 600; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; display: flex; align-items: center;">
                        <span style="background-color: #00A4EF; border-radius: 50%; width: 10px; height: 10px; display: inline-block; margin-right: 10px; box-shadow: 0 0 8px #00A4EF;"></span>
                        {name}
                    </h3>
                    <div style="display: flex; justify-content: space-between; font-size: 13px; color: #aaa; margin-bottom: 15px; background: #222; padding: 10px 15px; border-radius: 6px; border: 1px solid #2d2d2d;">
                        <div style="flex: 1;"><strong>Status:</strong> <span style="color: {'#4CAF50' if status == 'active' else '#FFC107'}; text-transform: uppercase; font-size: 11px;">{status}</span></div>
                        <div style="flex: 1;"><strong>Speed:</strong> <span style="color: #fff;">{speed_str}</span></div>
                        <div style="flex: 1;"><strong>Peers:</strong> <span style="color: #fff;">{peers}</span></div>
                        <div style="flex: 1;"><strong>ETA:</strong> <span style="color: #fff;">{eta_str}</span></div>
                    </div>
                    <div style="background-color: #0a0a0a; width: 100%; height: 24px; border-radius: 12px; overflow: hidden; position: relative; border: 1px solid #333;">
                        <div style="background: linear-gradient(90deg, #005A9E, #00A4EF); width: {progress}%; height: 100%; transition: width 0.3s ease;"></div>
                        <div style="position: absolute; width: 100%; top: 0; text-align: center; font-size: 12px; font-weight: bold; color: #fff; line-height: 24px; text-shadow: 1px 1px 3px #000;">
                            {progress:.1f}% &nbsp;&nbsp;|&nbsp;&nbsp; {downloaded_str} / {total_size_str}
                        </div>
                    </div>
                </div>
                <div style="max-height: 400px; overflow-y: auto; background-color: #0f0f0f; padding: 10px 15px;">
                    <table style="width: 100%; border-collapse: separate; border-spacing: 0 4px; font-size: 12px; text-align: left; color: #ccc;">
                        <thead style="position: sticky; top: -10px; background-color: #0f0f0f; z-index: 1;">
                            <tr style="color: #888; font-weight: normal;">
                                <th style="padding: 10px 10px 5px 10px; border-bottom: 1px solid #333;">File Name</th>
                                <th style="padding: 10px 10px 5px 10px; border-bottom: 1px solid #333; width: 100px; text-align: right;">Size</th>
                                <th style="padding: 10px 10px 5px 10px; border-bottom: 1px solid #333; width: 150px; text-align: right;">Progress</th>
                            </tr>
                        </thead>
                        <tbody>
                            {file_rows_html}
                        </tbody>
                    </table>
                </div>
            </div>
            """
            clear_output(wait=True)
            display(HTML(ui_html))
            time.sleep(1.5)

        # 4. TRANSFER TO DRIVE PHASE
        if download.is_complete:
            # Kill aria2c to release any file locks
            subprocess.run(["pkill", "-f", "aria2c"])

            if LOCAL_TMP_PATH != DOWNLOAD_PATH:
                print(f"✅ Local Download Finished! Preparing Google Drive Transfer...")

                # Get total size of files to move
                total_size = 0
                for dirpath, _, filenames in os.walk(LOCAL_TMP_PATH):
                    for f in filenames:
                        fp = os.path.join(dirpath, f)
                        if not os.path.islink(fp):
                            total_size += os.path.getsize(fp)

                if total_size > 0:
                    copied_size = 0
                    start_time = time.time()
                    last_ui_time = 0

                    for src_dir, dirs, files in os.walk(LOCAL_TMP_PATH):
                        # Create equivalent directory structure in Google Drive
                        dst_dir = src_dir.replace(LOCAL_TMP_PATH, DOWNLOAD_PATH, 1)
                        os.makedirs(dst_dir, exist_ok=True)

                        for file_ in files:
                            src_file = os.path.join(src_dir, file_)
                            dst_file = os.path.join(dst_dir, file_)

                            # Chunked copy to show progress
                            with open(src_file, 'rb') as fsrc, open(dst_file, 'wb') as fdst:
                                while True:
                                    buf = fsrc.read(1024 * 1024 * 8) # 8MB chunks for fast transfer
                                    if not buf:
                                        break
                                    fdst.write(buf)
                                    copied_size += len(buf)

                                    # Update UI
                                    now = time.time()
                                    if now - last_ui_time > 0.5:
                                        last_ui_time = now
                                        progress = (copied_size / total_size * 100)
                                        speed = copied_size / (now - start_time) if now > start_time else 0

                                        upload_html = f"""
                                        <div style="background-color: #121212; color: #f5f5f5; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; width: 100%; max-width: 800px; border: 1px solid #333; border-radius: 12px; padding: 25px; box-shadow: 0 8px 16px rgba(0,0,0,0.5);">
                                            <h3 style="color: #fff; margin-top: 0; display: flex; align-items: center;">
                                                <span style="background-color: #FF8C00; border-radius: 50%; width: 12px; height: 12px; display: inline-block; margin-right: 12px; box-shadow: 0 0 10px #FF8C00;"></span>
                                                ☁️ Uploading {download.name} to Google Drive...
                                            </h3>
                                            <div style="margin-bottom: 15px; font-size: 14px; color: #bbb; display: flex; justify-content: space-between; background: #1a1a1a; padding: 12px; border-radius: 6px; border: 1px solid #222;">
                                                <div><strong>Progress:</strong> <span style="color: #fff;">{progress:.1f}%</span></div>
                                                <div><strong>Transfer Speed:</strong> <span style="color: #fff;">{format_bytes(speed)}/s</span></div>
                                                <div><strong>Copied:</strong> <span style="color: #fff;">{format_bytes(copied_size)} / {format_bytes(total_size)}</span></div>
                                            </div>
                                            <div style="background-color: #0a0a0a; width: 100%; height: 24px; border-radius: 12px; overflow: hidden; border: 1px solid #333; position: relative;">
                                                <div style="background: linear-gradient(90deg, #FF8C00, #FFD700); width: {progress}%; height: 100%; transition: width 0.3s ease;"></div>
                                                <div style="position: absolute; width: 100%; top: 0; text-align: center; font-size: 12px; font-weight: bold; color: #fff; line-height: 24px; text-shadow: 1px 1px 3px #000;">
                                                    {progress:.1f}%
                                                </div>
                                            </div>
                                            <p style="color: #888; font-size: 11px; margin-bottom: 0; margin-top: 15px; text-align: center;">Colab local files are instantly deleted after copying to save space!</p>
                                        </div>
                                        """
                                        clear_output(wait=True)
                                        display(HTML(upload_html))

                            # ✨ CRITICAL FIX: Delete the source file immediately after it copies!
                            # This prevents the 62GB Colab disk from filling up, since we don't hold double the data.
                            os.remove(src_file)

                # Clean up empty local directories
                shutil.rmtree(LOCAL_TMP_PATH, ignore_errors=True)

            clear_output(wait=True)
            print("✅ Process Completely Finished!")
            print(f"📦 Name: {download.name}")
            print(f"📁 All files successfully saved to: {DOWNLOAD_PATH}")
        elif download.has_failed:
            print(f"❌ Download failed! Error: {download.error_message}")

OSError: [Errno 28] No space left on device

.

  .

.

.

.

In [ ]:
#@title 🚀 Open Google Drive in a New Tab (Attempt Auto-Open)
#@markdown Click Here and then click on output link to open your Google Drive

from IPython.display import display, HTML

drive_url = "https://drive.google.com/drive/u/0/my-drive" # The standard Google Drive URL
html_code = f'''
<a href="{drive_url}" target="_blank">Click here to open your Google Drive (if it doesn't open automatically)</a>
'''
display(HTML(html_code))